# Pose estimation

In [ ]:
"""Extract features for temporal action detection datasets

Adapated from https://github.com/m-bain/whisperX and https://huggingface.co/intfloat/multilingual-e5-large. 

Sam Perochon, December 2023.

"""
%precision 2
%load_ext autoreload
%autoreload 2
    
import torch.nn.functional as F
import json
import os
import sys
import numpy as np
import pandas as pd
import subprocess
import socket

from torch import Tensor
import logging
from  tqdm import tqdm
import time
import matplotlib.pyplot as plt 
import cv2

from decord import VideoReader, bridge, cpu, gpu

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

#@markdown To better demonstrate the Pose Landmarker API, we have created a set of visualization tools that will be used in this colab. These will draw the landmarks on a detect person, as well as the expected connections between those markers.

from mediapipe import solutions
from mediapipe.framework.formats import landmark_pb2
import numpy as np


def draw_landmarks_on_image(rgb_image, detection_result):
  pose_landmarks_list = detection_result.pose_landmarks
  annotated_image = np.copy(rgb_image)

  # Loop through the detected poses to visualize.
  for idx in range(len(pose_landmarks_list)):
    pose_landmarks = pose_landmarks_list[idx]

    # Draw the pose landmarks.
    pose_landmarks_proto = landmark_pb2.NormalizedLandmarkList()
    pose_landmarks_proto.landmark.extend([
      landmark_pb2.NormalizedLandmark(x=landmark.x, y=landmark.y, z=landmark.z) for landmark in pose_landmarks
    ])
    solutions.drawing_utils.draw_landmarks(
      annotated_image,
      pose_landmarks_proto,
      solutions.pose.POSE_CONNECTIONS,
      solutions.drawing_styles.get_default_pose_landmarks_style())
  return annotated_image

from main import get_data_root, fetch_output_path, fetch_video_paths, parse_flag

bridge.set_bridge('torch')

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# Set pandas default printing 
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 5000)
    

logging.basicConfig(filename=os.path.join(get_data_root(), 'log','skeleton_estimation_computation.log'),
                    filemode='a',
                    format='%(asctime)s,%(msecs)d %(name)s %(levelname)s %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S',
                    level=logging.DEBUG)

logging.info("Skeleton estimation computation")
logger = logging.getLogger('main')


# -------- 
# Define model and video dataset path

model_name = 'skeleton_landmarks_mediapipe'
model_path = '/home/sam/temporal_segmentation/smartflat/features/skeleton/model/pose_landmarker_heavy.task'
representation_name = 'skeleton_landmarks'
metadata_path = os.path.join(get_data_root(), 'dataframes', 'persistent_metadata', '{}_dataset_df.csv'.format(socket.gethostname()))
metrics_path = os.path.join(get_data_root(), 'dataframes', 'compute_time_skeleton_landmarks_detection.csv')
logging_interval = 2000



BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Create a pose landmarker instance with the video mode:
options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO, 
    output_segmentation_masks=True,
    num_poses=4
    )

In [ ]:
video_paths = fetch_video_paths(metadata_path, model_name)


In [ ]:
video_paths = ['/diskA/sam_data/data/gateau_2022/G98_P84_GIBAnt_07102022/Tobii/SGOT8NywPLjIn-w1C1rMxQ==.mp4']#video_paths[:1]
video_paths

In [ ]:
sampling_frequency = 1/2 # Estimate skeleton every half a second (multiplied by the fps of the video in practice)
logging_interval = 2000
plotting_interval = 500
verbose=True
import traceback

# Process..
metrics = [] 
        
for j, video_path in enumerate(reversed(video_paths)):
    
    start_time = time.time()
    print("Extracting skeleton landmarks {}/{} for {}".format(j+1, len(video_paths), video_path))
    
    skeleton_landmark_output_path = fetch_output_path(video_path, model_name)
    if os.path.isfile(skeleton_landmark_output_path) and (parse_flag(skeleton_landmark_output_path, 'skeleton_landmarks_representation') == 'success'):
        
        print("Computation aready done.")
        continue

        #continue

    try:
        vr = VideoReader(video_path, num_threads=5, ctx=cpu(0))

    except Exception as e:
        print(e)
        print('Error /!\ ')
        traceback.print_exc()
        
        with open(os.path.join(os.path.dirname(skeleton_landmark_output_path), f'.{os.path.basename(video_path)[:-4]}_skeleton_landmarks_flag.txt'), 'w') as f:
            f.write('failure')
        print("Wrote failure flags.", skeleton_landmark_output_path)
        continue


    
    fps =  vr.get_avg_fps(); n_frames =  vr._num_frame; duration = n_frames/fps
    results = []
    with PoseLandmarker.create_from_options(options) as landmarker:
            
        for i, idx in enumerate(tqdm(range(0, len(vr), int(fps * sampling_frequency)))):
    
            image = vr[idx].numpy()
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image)
            frame_timestamp_ms = int(idx / n_frames * duration * 1e3)
            
            # Perform hand landmarks detection on the provided single image.
            pose_landmarker_result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)
            results.append(pose_landmarker_result)

            if i % logging_interval == 0:
                logger.info("{}/{} {} - {}/{} [{:.2f} %]".format(j, len(video_paths), video_path, i, len(vr), 100*i/len(vr)))


            if verbose and i % plotting_interval == 0  and len(pose_landmarker_result.pose_landmarks) > 0:
        
                annotated_image = draw_landmarks_on_image(mp_image.numpy_view(), pose_landmarker_result)
                
                segmentation_mask = pose_landmarker_result.segmentation_masks[0].numpy_view()
                visualized_mask = np.repeat(segmentation_mask[:, :, np.newaxis], 3, axis=2) * 255
            
            
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5))
                
                ax1.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR))
                ax2.imshow(visualized_mask)
                plt.tight_layout()
                plt.show()


    # Success flag
    with open(os.path.join(os.path.dirname(skeleton_landmark_output_path), f'.{os.path.basename(video_path)[:-4]}_skeleton_landmarks_flag.txt'), 'w') as f:
        f.write('success')
        print("Wrote success flags: {}".format(os.path.join(os.path.dirname(skeleton_landmark_output_path), f'.{os.path.basename(video_path)[:-4]}_skeleton_landmarks_flag.txt')))
    
    # Save results and metrics
    with open(skeleton_landmark_output_path, 'w') as f:
        json.dump(results, f, default=lambda x: x if type(x) == list else x.__dict__)
        
    metrics.append({'video_path': video_path, 'num_frames': len(vr), 'hand_landmarks_compute_time': time.time() - start_time, 'sampling_frequency': sampling_frequency})
    if os.path.isfile(metrics_path):
        metricsdf = pd.concat([pd.read_csv(metrics_path), pd.DataFrame(metrics)])
    else:
        metricsdf =  pd.DataFrame(metrics)
    metricsdf.to_csv(metrics_path, index=False)

In [ ]:
import os
os._exit(00)

In [ ]:
annotated_image = draw_landmarks_on_image(mp_image.numpy_view(), pose_landmarker_result)

segmentation_mask = pose_landmarker_result.segmentation_masks[0].numpy_view()
visualized_mask = np.repeat(segmentation_mask[:, :, np.newaxis], 3, axis=2) * 255


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5))

ax1.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR))
ax2.imshow(visualized_mask)

In [ ]:
import cv2
annotated_image = draw_landmarks_on_image(mp_image.numpy_view(), pose_landmarker_result)

segmentation_mask = pose_landmarker_result.segmentation_masks[1].numpy_view()
visualized_mask = np.repeat(segmentation_mask[:, :, np.newaxis], 3, axis=2) * 255


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 12))

ax1.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR))
ax2.imshow(visualized_mask)

In [ ]:
logging_interval = 2000
# Process..
metrics = [] 
for j, video_path in enumerate(video_paths):
    
    start_time = time.time()
    print("Extracting skeleton landmarks {}/{} for {}".format(j+1, len(video_paths), video_path))
    
    skeleton_landmark_output_path = fetch_output_path(video_path, model_name)
    if os.path.isfile(skeleton_landmark_output_path):
        print("Computation aready done.")
        continue

    try:
        vr = VideoReader(video_path, num_threads=5, ctx=cpu(0))
    except:
        print('/!\ Aborted', video_path)
        
    fps =  vr.get_avg_fps(); n_frames =  vr._num_frame; duration = n_frames/fps
    
    results = []
    with PoseLandmarker.create_from_options(options) as landmarker:
            
        for i, idx in enumerate(tqdm(range(len(vr)))):
    
            image = vr[idx].numpy()
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image)
            frame_timestamp_ms = int(idx / n_frames * duration * 1e3)
            
            # Perform hand landmarks detection on the provided single image.
            pose_landmarker_result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)
            results.append(pose_landmarker_result)

            if i % logging_interval == 0:
                logger.info("{}/{} {} - {}/{} [{:.2f} %]".format(j, len(video_paths), video_path, i, len(vr), 100*i/len(vr)))

    # Save results and metrics
    with open(skeleton_landmark_output_path, 'w') as f:
        json.dump(results, f, default=lambda x: x if type(x) == list else x.__dict__)
        
    metrics.append({'video_path': video_path, 'num_frames': len(vr), 'hand_landmarks_compute_time': time.time() - start_time})
    if os.path.isfile(metrics_path):
        metricsdf = pd.concat([pd.read_csv(metrics_path), pd.DataFrame(metrics)])
    else:
        metricsdf =  pd.DataFrame(metrics)
    metricsdf.to_csv(metrics_path, index=False)